> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board and uploads the
> package. *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host,
>   writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python
(CPython, the project `.venv`) on this computer. After `%board connect`, cells run in
MicroPython on the board, except cells that start with `%%host`.

Notebook-only: import the package before anything else. Its `__init__` imports `crc`, and
the header cell below rewrites `crc.py`; importing the package first means no later import
reads the half-written file.

In [ ]:
import micropython_i2c  # noqa: F401

# Hardware and configuration

| | Host run | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | none | none: `crc` is pure arithmetic and never touches the bus |
| **Result** | not run yet on the hardware | not run yet |

# TODO:
- **Widths below 8 bits give wrong checksums** (found 2026-11-16, not fixed: the API and
  behaviour are frozen until decided). `_crc_align` shifts the input byte *right* when
  `CRC_WIDTH < 8`, which drops input bits, and the table path indexes with the whole byte.
  CRC-7/MMC over `b"123456789"` gives `0x33` bitwise and `0x2F` with the table; the catalogue
  check value is `0x75`. No driver uses a width below 8 (the parts on the bench are all
  CRC-8). Widths of 8 and above, including 10 and 12, match the catalogue.

Decided (2026-11-16): `crc.py` moved from hand-written to this notebook. The API and
behaviour are unchanged; the tests are the ones from the hand-written `tests/test_crc.py`
(ported from the old single-file notebook's sections 10.5 to 10.11), one per cell.

# Python file
location: `src/micropython_i2c/crc.py`

tests: `tests/test_crc.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/crc.py
"""``CRCMixin``: arbitrary-width CRC, parameterised entirely by class attributes.

The parameters are datasheet facts about a part, so they live on the driver class the
same way ``ALLOWED_ADDRS`` does. A module-level ``crc8(data, poly, init, ...)`` would
push them to every call site, where they would drift.

Reading the polynomial off a datasheet: drop the x^n term and set a bit per remaining
power. x^8+x^2+x+1 -> ``0x07``; x^8+x^5+x^4+1 -> ``0x31``; x^16+x^15+x^2+1 -> ``0x8005``.

``CRC_POLY`` deliberately has no default. A wrong polynomial yields checksums that
verify against themselves and nothing else, so an unset one raises.

What bytes a checksum covers is the *driver's* decision, not this class's. Sensirion
parts checksum each 16-bit word; the TMAG5273 checksums the whole packet including the
command byte the host sent.

Widths below 8 bits are not supported: the byte alignment drops input bits there.

Generated by ``dev_notebooks/crc_devNB``: edit the notebook, not this file.
"""

# Imports
## External
None.

## Internal
`BitTools`, the base class: reflected CRCs need `bit_reverse`, and a driver that mixes in
`CRCMixin` gets the rest of the bit helpers with it.

In [ ]:
%%extract ../src/micropython_i2c/crc.py -a
from micropython_i2c.bits import BitTools

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run. `CommandI2CBase` is there for
one test: the Sensirion protocol mixin's CRC defaults must give the SHTC3 datasheet values.

In [ ]:
%%writefile ../tests/test_crc.py
"""CRCMixin, offline.

Vectors come from three independent sources: the SHTC3 datasheet (Table 16), the
TMAG5273 datasheet (section 7.4.1, clean page render) and the standard ``123456789``
check values for four catalogued variants spanning both widths and both reflections.
"""

import pytest

from micropython_i2c.bits import BitTools
from micropython_i2c.crc import CRCMixin
from micropython_i2c.protocol import CommandI2CBase

Notebook-only: the imports the validation and test cells use. `BitTools` and `CRCMixin` come
from the code cells below.

In [ ]:
import pytest

from micropython_i2c.protocol import CommandI2CBase

# Intent
**Purpose:** one CRC implementation for every part that checksums its data, configured by
the part's datasheet facts (width, polynomial, initial value, reflection, final XOR) declared
as class attributes on the driver.

**Why class attributes:** a part's CRC parameters never change, so they belong with its other
datasheet facts (`ALLOWED_ADDRS`, register map). A module function `crc8(data, poly, init)`
would repeat them at every call site, where one would eventually be wrong.

**Why `CRC_POLY` has no default:** a wrong polynomial gives checksums that verify against
themselves and nothing else, so tests on the host would pass and the real part would fail.
An unset polynomial raises `NotImplementedError` naming the class.

The model is the usual parameterised ("Rocksoft") CRC:

```mermaid
flowchart LR
    I[CRC_INIT] --> L{for each byte}
    L -->|CRC_REFLECT_IN: bit_reverse| X[XOR into the top of the accumulator]
    X --> S[8 shift/XOR steps with CRC_POLY, or one table lookup]
    S --> L
    L -->|done| O[CRC_REFLECT_OUT: bit_reverse] --> F[XOR CRC_XOR_OUT] --> R[checksum]
```

| Part | `CRC_WIDTH` | `CRC_POLY` | `CRC_INIT` | Covers |
|---|---|---|---|---|
| SHTC3 (Sensirion) | 8 | `0x31` ($x^8+x^5+x^4+1$) | `0xFF` | each 16-bit word |
| TMAG5273 | 8 | `0x07` ($x^8+x^2+x+1$) | `0xFF` | the whole packet, command byte included |

**Used by:** `CommandI2CBase` (Sensirion defaults), the SHTC3 and TMAG5273 drivers.

**Not its job:** deciding which bytes a checksum covers. That is the driver's.

## What the module defines

| Name | What it is |
|---|---|
| `CRCMixin` | the CRC model, a `BitTools` subclass configured by `CRC_*` class attributes |
| `CRCMixin.crc` | the checksum of some bytes, optionally resuming from `init` |
| `CRCMixin.crc_ok` | `True` if a checksum matches |
| `CRCMixin.crc_verify` | the checksum, or `ValueError` if it does not match |
| `CRCMixin.crc_table`, `crc_table_clear` | the lazily built 256-entry lookup table |
| `CRCMixin.crc_mask`, `crc_topbit` | width-derived constants |
| `CRCMixin.crc_check_configured` | raise if `CRC_POLY` is unset |

# Code
## TODO:
Nothing (the width-below-8 bug is in the top `# TODO:`).

## Public names

In [ ]:
%%extract ../src/micropython_i2c/crc.py -a
__all__ = ["CRCMixin"]

## `CRCMixin`
Every method is a `classmethod`: the parameters are class attributes, and a checksum never
needs an instance (a driver's `self.crc(...)` works the same way). Derived values
(`crc_mask`, `crc_topbit`) are methods rather than attributes so they follow a subclass that
overrides `CRC_WIDTH`.

| Attribute | Default | Meaning |
|---|---|---|
| `CRC_WIDTH` | `8` | checksum width in bits |
| `CRC_POLY` | `None` | generator polynomial without the $x^n$ term; must be set |
| `CRC_INIT` | `0xFF` | starting accumulator |
| `CRC_REFLECT_IN` | `False` | bit-reverse each input byte |
| `CRC_REFLECT_OUT` | `False` | bit-reverse the final accumulator |
| `CRC_XOR_OUT` | `0x00` | XORed into the result last |
| `CRC_TABLE_ENABLE` | `False` | use a 256-entry table: about 8 times fewer operations per byte, 256 entries of RAM per parameter set |

The table is cached in the class's own `__dict__`, not found by inherited lookup, so a
subclass that changes a parameter builds its own table instead of reusing its parent's.
After changing a parameter on a class that already built one, call `crc_table_clear()`.

`crc_verify` raises rather than returning a flag on purpose: a corrupted reading looks
exactly like a real one, so an unchecked `False` is a live risk on a data path.

In [ ]:
%%extract ../src/micropython_i2c/crc.py -a
class CRCMixin(BitTools):
    """Full parameterised CRC model: width, poly, init, reflect in/out, final XOR.

    Class attributes:
        CRC_WIDTH: Checksum width in bits (8 and 16 are usual; below 8 is not supported).
        CRC_POLY: Generator polynomial with the x^n term omitted. Must be set.
        CRC_INIT: Starting accumulator value.
        CRC_REFLECT_IN: Bit-reverse each input byte.
        CRC_REFLECT_OUT: Bit-reverse the final accumulator.
        CRC_XOR_OUT: XORed into the result last.
        CRC_TABLE_ENABLE: Use a lazily built 256-entry table (~8x fewer ops per byte,
            256 entries of RAM per parameter set). Worth it only on hot polling paths.
    """

    CRC_WIDTH = 8
    CRC_POLY = None
    CRC_INIT = 0xFF
    CRC_REFLECT_IN = False
    CRC_REFLECT_OUT = False
    CRC_XOR_OUT = 0x00
    CRC_TABLE_ENABLE = False

    # Derived values are classmethods, not attributes, so they track a subclass that
    # overrides CRC_WIDTH.

    @classmethod
    def crc_mask(cls):
        """All-ones mask for ``CRC_WIDTH``."""
        return (1 << cls.CRC_WIDTH) - 1

    @classmethod
    def crc_topbit(cls):
        """The bit that shifts out of the accumulator and triggers the XOR."""
        return 1 << (cls.CRC_WIDTH - 1)

    @classmethod
    def crc_check_configured(cls):
        """Raise ``NotImplementedError`` naming the class if ``CRC_POLY`` is unset."""
        if cls.CRC_POLY is None:
            raise NotImplementedError(
                "{}: CRC_POLY not set; declare the polynomial from the datasheet "
                "(x^8+x^2+x+1 -> 0x07, x^8+x^5+x^4+1 -> 0x31)".format(cls.__name__)
            )

    @classmethod
    def _crc_step8(cls, crc, mask, topbit):
        """Run the eight shift/XOR steps for one byte already folded into ``crc``."""
        poly = cls.CRC_POLY
        for _ in range(8):
            crc = ((crc << 1) ^ poly) & mask if crc & topbit else (crc << 1) & mask
        return crc

    @classmethod
    def _crc_align(cls, byte):
        """Align an input byte with the top of the accumulator."""
        shift = cls.CRC_WIDTH - 8
        return byte << shift if shift >= 0 else byte >> -shift

    @classmethod
    def crc_table(cls):
        """Return the 256-entry lookup table, building it on first use.

        Cached in ``cls.__dict__`` (not by inherited lookup) so a subclass that narrows
        the parameters builds its own table instead of reusing its parent's.

        Raises:
            NotImplementedError: If ``CRC_POLY`` is unset.
        """
        cached = cls.__dict__.get("_crc_tbl")
        if cached is not None:
            return cached
        cls.crc_check_configured()
        mask, topbit = cls.crc_mask(), cls.crc_topbit()
        table = [cls._crc_step8(cls._crc_align(b), mask, topbit) for b in range(256)]
        cls._crc_tbl = table
        return table

    @classmethod
    def crc_table_clear(cls):
        """Drop this class's cached table; needed after changing a CRC parameter."""
        if "_crc_tbl" in cls.__dict__:
            del cls._crc_tbl

    @classmethod
    def crc(cls, data, init=None):
        """Checksum over ``data`` using the class parameters.

        Args:
            data: Bytes to checksum.
            init: Override the starting value, to resume a running checksum across
                transfers. Reflection and final XOR apply at the end of every call, so
                resuming only works with those at their identity values.

        Returns:
            The checksum, ``CRC_WIDTH`` bits wide.

        Raises:
            NotImplementedError: If ``CRC_POLY`` is unset.
        """
        cls.crc_check_configured()
        mask, topbit = cls.crc_mask(), cls.crc_topbit()
        shift = cls.CRC_WIDTH - 8
        crc = cls.CRC_INIT if init is None else init
        table = cls.crc_table() if cls.CRC_TABLE_ENABLE else None

        for byte in data:
            if cls.CRC_REFLECT_IN:
                byte = cls.bit_reverse(byte, 8)
            if table is None:
                crc = cls._crc_step8(crc ^ cls._crc_align(byte), mask, topbit)
            elif shift > 0:
                # Index by top byte of the accumulator XOR input; shift the rest up.
                crc = table[((crc >> shift) ^ byte) & 0xFF] ^ ((crc << 8) & mask)
            else:
                crc = table[(crc ^ byte) & 0xFF]

        if cls.CRC_REFLECT_OUT:
            crc = cls.bit_reverse(crc, cls.CRC_WIDTH)
        return (crc ^ cls.CRC_XOR_OUT) & mask

    @classmethod
    def crc_ok(cls, data, expected, init=None):
        """True if the checksum over ``data`` equals ``expected``.

        Prefer ``crc_verify`` on a data path, where an unchecked boolean is a live risk.

        Raises:
            NotImplementedError: If ``CRC_POLY`` is unset.
        """
        return cls.crc(data, init=init) == expected

    @classmethod
    def crc_verify(cls, data, expected, context="", init=None):
        """Return the checksum, or raise ``ValueError`` if it does not match ``expected``.

        Raising rather than returning a flag is deliberate: a corrupted reading looks
        exactly like a real one. ``context`` (a word index, a register name) is
        prepended to the message.

        Raises:
            ValueError: If the checksum over ``data`` is not ``expected``.
            NotImplementedError: If ``CRC_POLY`` is unset.
        """
        actual = cls.crc(data, init=init)
        if actual != expected:
            raise ValueError(
                "{}checksum fail: computed {:#04x}, received {:#04x}".format(
                    context + ": " if context else "", actual, expected
                )
            )
        return actual

### validation/test
The parts and catalogue variants the checks below use, shared with the test file.
`crc_class` makes a throw-away `CRCMixin` subclass with the given parameters, the same way a
driver declares them.

| Set | Source of the expected values |
|---|---|
| `SHTC3` | SHTC3 datasheet, Table 16: `0x00` -> `0xAC`, `0xBEEF` -> `0x92` |
| `TMAG` | TMAG5273 datasheet, section 7.4.1 |
| `CRC-8/SMBUS`, `CRC-8/MAXIM`, `CRC-16/CCITT-FALSE`, `CRC-16/ARC` | the catalogue check value over `b"123456789"` |

In [ ]:
%%extract ../tests/test_crc.py -a
def crc_class(name="C", **params):
    return type(name, (CRCMixin,), params)


SHTC3 = crc_class("SHTC3", CRC_POLY=0x31, CRC_INIT=0xFF)
TMAG = crc_class("TMAG", CRC_POLY=0x07, CRC_INIT=0xFF)

PARAM_SETS = {
    "CRC-8/SMBUS": dict(CRC_WIDTH=8, CRC_POLY=0x07, CRC_INIT=0x00),
    "SHTC3": dict(CRC_WIDTH=8, CRC_POLY=0x31, CRC_INIT=0xFF),
    "TMAG5273": dict(CRC_WIDTH=8, CRC_POLY=0x07, CRC_INIT=0xFF),
    "CRC-8/MAXIM": dict(CRC_WIDTH=8, CRC_POLY=0x31, CRC_INIT=0x00,
                        CRC_REFLECT_IN=True, CRC_REFLECT_OUT=True),
    "CRC-16/CCITT-FALSE": dict(CRC_WIDTH=16, CRC_POLY=0x1021, CRC_INIT=0xFFFF),
    "CRC-16/ARC": dict(CRC_WIDTH=16, CRC_POLY=0x8005, CRC_INIT=0x0000,
                       CRC_REFLECT_IN=True, CRC_REFLECT_OUT=True),
}  # fmt: skip
CHECK_VALUES = {"CRC-8/SMBUS": 0xF4, "CRC-8/MAXIM": 0xA1, "CRC-16/CCITT-FALSE": 0x29B1,
                "CRC-16/ARC": 0xBB3D}  # fmt: skip
CORPUS = (b"\x00", b"\xff", b"\x12\x34", b"\xbe\xef", b"123456789", b"\x00" * 4, b"\xff" * 4,
          bytes(range(32)), bytes(range(255, 223, -1)))  # fmt: skip

#### validation: nominal usage
The datasheet vectors, the catalogue check values (bitwise and with the table), and a
`crc_verify` that passes.

In [ ]:
print("SHTC3 0x00 -> {:#04x} | 0xBEEF -> {:#04x}".format(SHTC3.crc(b"\x00"), SHTC3.crc(b"\xbe\xef")))
print("TMAG  0xE0 -> {:#04x}".format(TMAG.crc(b"\xe0")))
for name in sorted(CHECK_VALUES):
    bitwise = crc_class(**PARAM_SETS[name]).crc(b"123456789")
    tabled = crc_class(**PARAM_SETS[name], CRC_TABLE_ENABLE=True).crc(b"123456789")
    print("{:<19} {:#06x} | table {:#06x} | catalogue {:#06x}".format(name, bitwise, tabled, CHECK_VALUES[name]))
print("crc_verify:", hex(SHTC3.crc_verify(b"\xbe\xef", 0x92)))

#### validation: anti-nominal usage
A class without `CRC_POLY` refuses to checksum.

In [ ]:
try:
    crc_class("Unconfigured").crc(b"\x00")  # CRC_POLY was never declared
except NotImplementedError as err:
    print("raised NotImplementedError, as expected:\n    {}".format(err))
else:
    raise AssertionError("crc() without CRC_POLY should have raised NotImplementedError")

A checksum that does not match raises `ValueError`, with the caller's context first.

In [ ]:
try:
    SHTC3.crc_verify(b"\xbe\xef", 0x00, context="word 3")  # 0x92 is the right checksum
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("crc_verify with a wrong checksum should have raised ValueError")

The width-below-8 bug from the `# TODO:`, shown rather than tested (a test would fail): the
bitwise and table results disagree, and neither is the catalogue value `0x75`.

In [ ]:
mmc = dict(CRC_WIDTH=7, CRC_POLY=0x09, CRC_INIT=0x00)
print("CRC-7/MMC: bitwise {:#04x} | table {:#04x} | catalogue 0x75".format(
    crc_class(**mmc).crc(b"123456789"),
    crc_class(**mmc, CRC_TABLE_ENABLE=True).crc(b"123456789")))  # fmt: skip

#### unit

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


def test_unset_polynomial_raises_everywhere():
    unset = crc_class("Unconfigured")
    for fn in (lambda: unset.crc(b"\x00"), lambda: unset.crc_ok(b"\x00", 0),
               lambda: unset.crc_verify(b"\x00", 0), unset.crc_table):  # fmt: skip
        with pytest.raises(NotImplementedError, match="Unconfigured"):
            fn()

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("data, want", [(b"\x00", 0xAC), (b"\xbe\xef", 0x92)])
def test_shtc3_vectors(data, want):
    assert SHTC3.crc(data) == want
    assert CommandI2CBase.crc(data) == want  # Sensirion defaults

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize(
    "data, want", [(b"\xe0", 0x5D), (b"\x00\x00\x00\x00", 0xD1), (b"\xff\xff\xff\xff", 0x0F)]
)
def test_tmag5273_vectors(data, want):
    assert TMAG.crc(data) == want

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("name", sorted(CHECK_VALUES))
def test_standard_check_value(name):
    assert crc_class(**PARAM_SETS[name]).crc(b"123456789") == CHECK_VALUES[name]

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


def test_width_masks():
    c8, c16 = crc_class(**PARAM_SETS["CRC-8/SMBUS"]), crc_class(**PARAM_SETS["CRC-16/ARC"])
    assert (c8.crc_mask(), c16.crc_mask()) == (0xFF, 0xFFFF)
    assert (c8.crc_topbit(), c16.crc_topbit()) == (0x80, 0x8000)

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


def test_parameters_in_isolation():
    base_params = dict(CRC_POLY=0x07, CRC_INIT=0xFF)
    data = b"\x12\x34\x56"
    base = crc_class(**base_params).crc(data)
    assert crc_class(**base_params, CRC_XOR_OUT=0xFF).crc(data) == base ^ 0xFF
    assert crc_class(**base_params, CRC_REFLECT_OUT=True).crc(data) == BitTools.bit_reverse(base)
    assert crc_class(**base_params, CRC_REFLECT_IN=True).crc(data) != base

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


def test_init_override_and_resume():
    data = b"\x12\x34\x56"
    whole = TMAG.crc(data)
    assert TMAG.crc(data, init=0x00) != whole
    assert TMAG.crc(data[2:], init=TMAG.crc(data[:2])) == whole

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("name", sorted(PARAM_SETS))
def test_table_matches_bitwise(name):
    bitwise = crc_class(**PARAM_SETS[name])
    tabled = crc_class(**PARAM_SETS[name], CRC_TABLE_ENABLE=True)
    for data in CORPUS:
        assert bitwise.crc(data) == tabled.crc(data)

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


def test_table_lazy_cached_and_clearable():
    tbl = crc_class(CRC_POLY=0x07, CRC_TABLE_ENABLE=True)
    assert "_crc_tbl" not in tbl.__dict__
    tbl.crc(b"\x00")
    assert len(tbl.crc_table()) == 256
    assert tbl.crc_table() is tbl.crc_table()
    tbl.crc_table_clear()
    assert "_crc_tbl" not in tbl.__dict__

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


def test_subclass_builds_own_table():
    parent = crc_class("Parent", CRC_POLY=0x07, CRC_INIT=0xFF, CRC_TABLE_ENABLE=True)
    child = type("Child", (parent,), {"CRC_POLY": 0x31})
    parent.crc(b"\x00")
    assert child.crc(b"\x00") == 0xAC
    assert parent.crc(b"\xe0") == 0x5D
    assert parent.crc_table() is not child.crc_table()

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


def test_ok_and_verify():
    good = SHTC3.crc(b"\xbe\xef")
    assert SHTC3.crc_ok(b"\xbe\xef", good)
    assert not SHTC3.crc_ok(b"\xbe\xef", good ^ 0xFF)
    assert SHTC3.crc_verify(b"\xbe\xef", good) == good
    with pytest.raises(ValueError, match="word 3"):
        SHTC3.crc_verify(b"\xbe\xef", 0x00, context="word 3")

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


def test_every_single_bit_flip_caught():
    good = SHTC3.crc(b"\xbe\xef")
    for bit in range(16):
        corrupt = (0xBEEF ^ (1 << bit)).to_bytes(2, "big")
        assert not SHTC3.crc_ok(corrupt, good)

In [ ]:
%%extract ../tests/test_crc.py -a --strip-ipytest
%%ipytest


def test_tmag_scope_includes_command_byte():
    cmd = bytes([(0x35 << 1) | 1])
    payload = bytes([0x12, 0x34, 0x56, 0x78, 0x10])
    crc = TMAG.crc(cmd + payload)
    assert TMAG.crc_ok(cmd + payload, crc)
    assert not TMAG.crc_ok(payload, crc)

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3 and uploads the
> package. Nothing here uses the I2C bus.
>
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

Layout: connect and set up, the **matching tests** (the same vectors as the unit tests
above, on MicroPython), then the tally and disconnect. The table checks matter most here:
the table cache relies on a class's `__dict__` and on deleting a class attribute, which
MicroPython implements differently from CPython.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's
folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c.bits import BitTools
from micropython_i2c.crc import CRCMixin
from micropython_i2c.debug import check, check_raises, check_reset, check_summary
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)
from micropython_i2c.protocol import CommandI2CBase

check_reset()

Notebook-only: the same parameter sets as the host's, and two board helpers. `crc_class`
builds its class dictionary by hand (no `**a, k=v` merging, which not every MicroPython build
accepts). `run` turns an exception into a `FAIL` line, so one broken check does not stop the
section before `%board disconnect`.

**What to do:** nothing.

In [ ]:
def crc_class(name="C", params=None, **extra):
    d = dict(params or {})
    d.update(extra)
    return type(name, (CRCMixin,), d)


def run(label, fn):
    try:
        ok = fn()
    except Exception as e:  # report it as a FAIL and go on
        return check(label, False, "raised {}: {}".format(type(e).__name__, e))
    return check(label, ok)


SHTC3 = crc_class("SHTC3", CRC_POLY=0x31, CRC_INIT=0xFF)
TMAG = crc_class("TMAG", CRC_POLY=0x07, CRC_INIT=0xFF)
PARAM_SETS = {
    "CRC-8/SMBUS": dict(CRC_WIDTH=8, CRC_POLY=0x07, CRC_INIT=0x00),
    "SHTC3": dict(CRC_WIDTH=8, CRC_POLY=0x31, CRC_INIT=0xFF),
    "TMAG5273": dict(CRC_WIDTH=8, CRC_POLY=0x07, CRC_INIT=0xFF),
    "CRC-8/MAXIM": dict(CRC_WIDTH=8, CRC_POLY=0x31, CRC_INIT=0x00,
                        CRC_REFLECT_IN=True, CRC_REFLECT_OUT=True),
    "CRC-16/CCITT-FALSE": dict(CRC_WIDTH=16, CRC_POLY=0x1021, CRC_INIT=0xFFFF),
    "CRC-16/ARC": dict(CRC_WIDTH=16, CRC_POLY=0x8005, CRC_INIT=0x0000,
                       CRC_REFLECT_IN=True, CRC_REFLECT_OUT=True),
}  # fmt: skip
CHECK_VALUES = {"CRC-8/SMBUS": 0xF4, "CRC-8/MAXIM": 0xA1, "CRC-16/CCITT-FALSE": 0x29B1,
                "CRC-16/ARC": 0xBB3D}  # fmt: skip
CORPUS = (b"\x00", b"\xff", b"\x12\x34", b"\xbe\xef", b"123456789", b"\x00" * 4, b"\xff" * 4,
          bytes(range(32)), bytes(range(255, 223, -1)))  # fmt: skip

## Matching tests

**Matches** `test_unset_polynomial_raises_everywhere`.

**What to do:** nothing.

In [ ]:
unset = crc_class("Unconfigured")
check_raises("unset CRC_POLY: crc", NotImplementedError, unset.crc, b"\x00")
check_raises("unset CRC_POLY: crc_ok", NotImplementedError, unset.crc_ok, b"\x00", 0)
check_raises("unset CRC_POLY: crc_verify", NotImplementedError, unset.crc_verify, b"\x00", 0)
check_raises("unset CRC_POLY: crc_table", NotImplementedError, unset.crc_table)

**Matches** `test_shtc3_vectors`, `test_tmag5273_vectors` and `test_standard_check_value`.

**What to do:** nothing.

In [ ]:
for data, want in ((b"\x00", 0xAC), (b"\xbe\xef", 0x92)):
    got = SHTC3.crc(data)
    check("test_shtc3_vectors[{}]".format(data), got == want == CommandI2CBase.crc(data),
          "{:#04x}".format(got))  # fmt: skip
for data, want in ((b"\xe0", 0x5D), (b"\x00\x00\x00\x00", 0xD1), (b"\xff\xff\xff\xff", 0x0F)):
    got = TMAG.crc(data)
    check("test_tmag5273_vectors[{}]".format(data), got == want, "{:#04x}".format(got))
for name in sorted(CHECK_VALUES):
    got = crc_class(params=PARAM_SETS[name]).crc(b"123456789")
    check("test_standard_check_value[{}]".format(name), got == CHECK_VALUES[name],
          "{:#06x}".format(got))  # fmt: skip

**Matches** `test_width_masks`, `test_parameters_in_isolation` and
`test_init_override_and_resume`.

**What to do:** nothing.

In [ ]:
c8, c16 = crc_class(params=PARAM_SETS["CRC-8/SMBUS"]), crc_class(params=PARAM_SETS["CRC-16/ARC"])
check("test_width_masks",
      (c8.crc_mask(), c16.crc_mask()) == (0xFF, 0xFFFF)
      and (c8.crc_topbit(), c16.crc_topbit()) == (0x80, 0x8000))  # fmt: skip
base_params = dict(CRC_POLY=0x07, CRC_INIT=0xFF)
data = b"\x12\x34\x56"
base = crc_class(params=base_params).crc(data)
check("test_parameters_in_isolation",
      crc_class(params=base_params, CRC_XOR_OUT=0xFF).crc(data) == base ^ 0xFF
      and crc_class(params=base_params, CRC_REFLECT_OUT=True).crc(data) == BitTools.bit_reverse(base)
      and crc_class(params=base_params, CRC_REFLECT_IN=True).crc(data) != base)  # fmt: skip
whole = TMAG.crc(data)
check("test_init_override_and_resume",
      TMAG.crc(data, init=0x00) != whole and TMAG.crc(data[2:], init=TMAG.crc(data[:2])) == whole)  # fmt: skip

**Matches** `test_table_matches_bitwise`, `test_table_lazy_cached_and_clearable` and
`test_subclass_builds_own_table`: the table path on MicroPython's class dictionaries.

**What to do:** nothing. A `FAIL ... raised AttributeError` here means this MicroPython build
cannot read or delete a class's `__dict__` entries; `CRC_TABLE_ENABLE` must then stay off on
the board.

In [ ]:
for name in sorted(PARAM_SETS):
    bitwise = crc_class(params=PARAM_SETS[name])
    tabled = crc_class(params=PARAM_SETS[name], CRC_TABLE_ENABLE=True)
    run("test_table_matches_bitwise[{}]".format(name),
        lambda b=bitwise, t=tabled: all(b.crc(d) == t.crc(d) for d in CORPUS))  # fmt: skip


def table_lazy_cached_and_clearable():
    tbl = crc_class(CRC_POLY=0x07, CRC_TABLE_ENABLE=True)
    fresh = "_crc_tbl" not in tbl.__dict__
    tbl.crc(b"\x00")
    built = len(tbl.crc_table()) == 256 and tbl.crc_table() is tbl.crc_table()
    tbl.crc_table_clear()
    return fresh and built and "_crc_tbl" not in tbl.__dict__


def subclass_builds_own_table():
    parent = crc_class("Parent", CRC_POLY=0x07, CRC_INIT=0xFF, CRC_TABLE_ENABLE=True)
    child = type("Child", (parent,), {"CRC_POLY": 0x31})
    parent.crc(b"\x00")
    return (child.crc(b"\x00") == 0xAC and parent.crc(b"\xe0") == 0x5D
            and parent.crc_table() is not child.crc_table())  # fmt: skip


run("test_table_lazy_cached_and_clearable", table_lazy_cached_and_clearable)
run("test_subclass_builds_own_table", subclass_builds_own_table)

**Matches** `test_ok_and_verify`, `test_every_single_bit_flip_caught` and
`test_tmag_scope_includes_command_byte`.

**What to do:** nothing. The `ValueError` message is printed under its `PASS` line.

In [ ]:
good = SHTC3.crc(b"\xbe\xef")
check("test_ok_and_verify: crc_ok",
      SHTC3.crc_ok(b"\xbe\xef", good) and not SHTC3.crc_ok(b"\xbe\xef", good ^ 0xFF))  # fmt: skip
check("test_ok_and_verify: crc_verify passes", SHTC3.crc_verify(b"\xbe\xef", good) == good)
err = check_raises("test_ok_and_verify: crc_verify raises", ValueError,
                   SHTC3.crc_verify, b"\xbe\xef", 0x00, context="word 3")  # fmt: skip
check("test_ok_and_verify: context in the message", err is not None and "word 3" in str(err))
check("test_every_single_bit_flip_caught",
      not any(SHTC3.crc_ok((0xBEEF ^ (1 << bit)).to_bytes(2, "big"), good) for bit in range(16)))  # fmt: skip
cmd = bytes([(0x35 << 1) | 1])
payload = bytes([0x12, 0x34, 0x56, 0x78, 0x10])
crc = TMAG.crc(cmd + payload)
check("test_tmag_scope_includes_command_byte",
      TMAG.crc_ok(cmd + payload, crc) and not TMAG.crc_ok(payload, crc))  # fmt: skip

## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))